# TP 1 — Regresión Softmax para clasificación de texto (20 Newsgroups)

**Versión corregida.** Cambios respecto de la entrega anterior, según la devolución:

| Punto de la devolución | Dónde se resolvió |
|---|---|
| Faltaban las 10 preguntas de reflexión (sección 9) | Sección 9 al final |
| TensorBoard / MLflow importados pero sin uso; `train_softmax_model` sin registro | `train_softmax_model` (writer de TensorBoard, `mlflow.log_metric`, print por época); `objective` y entrenamiento final registran en ambos |
| Análisis de la búsqueda genérico, sin valores | Sección *Análisis de hiperparámetros*: tabla de trials y texto generado con los números de la corrida |
| Se describían 3 comportamientos pero se mostraban 2 trials | Sección *Dinámica*: un trial por cada comportamiento descripto (lr bajo, lr alto, bigramas + vocabulario grande), además del mejor y del peor; el texto sale de las métricas medidas |
| Curvas "Val" calculadas sobre test | Todas las curvas de dinámica usan la **validación** (split 80/20 de la búsqueda). El test se usa una sola vez, al final |
| Sin tiempos de convergencia | Sección *Tiempos de convergencia*: tiempo por época y tiempo hasta alcanzar un objetivo de accuracy, con experimento controlado |
| 10 trials en vez de 15 | `N_TRIALS = 15` |
| Explicación de código/frameworks breve | Sección *Explicación del código* ampliada (incluye TensorBoard y MLflow) |
| Peor trial reentrenado con otro split | Todos los trials de dinámica usan el mismo split que la búsqueda |
| Tiempos incluían vectorización y sin cifras | Se separan `prep_time` (TF-IDF + densificado) y `train_time`; el texto cita las cifras |
| Sin reporte por clase / matriz de confusión / curvas finales | Sección *Entrenamiento final*: reporte por clase, matriz de confusión, curvas guardadas en `figures/` y `results/` |
| Sin análisis de bias y varianza | Sección *Bias y varianza* |
| Entrenamiento final con test como `val_loader` y 15 épocas fijas | Las épocas se eligen con validación; el entrenamiento final no mira el test (`val_loader=None`) |

> **Cómo ejecutarlo:** *Run All*. Todos los números del texto se calculan en las celdas, así que se actualizan solos con cada corrida. Los experimentos con TF-IDF denso consumen bastante RAM (~1–2 GB con 10000 features).

In [ ]:
import re
import os
import time
import random
from functools import lru_cache
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import plotly
import optuna.visualization.matplotlib as vis
from IPython.display import display, Markdown


from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

from torch.utils.tensorboard import SummaryWriter
import mlflow
import mlflow.pytorch
import optuna

# Configuración de semillas y dispositivo (GPU/CPU)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)

# Carpetas de salida y tracking de experimentos
os.makedirs("figures", exist_ok=True)      # gráficos guardados
os.makedirs("results", exist_ok=True)      # tablas guardadas
mlflow.set_tracking_uri("sqlite:///mlflow.db")   # base local de MLflow (no se versiona)
mlflow.set_experiment("TP1_Softmax")
TB_DIR = "runs/tp1"                              # logs de TensorBoard: `tensorboard --logdir runs`

# Descarga de recursos de NLTK
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("punkt")
nltk.download("punkt_tab")

# Carga del dataset
newsgroups_train = fetch_20newsgroups(subset="train", remove=("headers", "footers", "quotes"))
newsgroups_test  = fetch_20newsgroups(subset="test",  remove=("headers", "footers", "quotes"))
class_names = newsgroups_train.target_names
NUM_CLASSES = len(class_names)
print(f"Clases ({NUM_CLASSES}):", class_names)

# Funciones de Preprocesamiento, Vectorización y Modelo

In [ ]:
STOP_WORDS = set(stopwords.words("english"))
STEMMER = PorterStemmer()
LEMMATIZER = WordNetLemmatizer()

# 1. Función de Preprocesamiento (TODO 1 Resuelto)
def preprocess_text(text, remove_stopwords, normalization):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha()]

    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOP_WORDS]

    if normalization == "stem":
        tokens = [STEMMER.stem(t) for t in tokens]
    elif normalization == "lemmatize":
        tokens = [LEMMATIZER.lemmatize(t) for t in tokens]

    return " ".join(tokens)

# 2. Función de Vectorización (TODO 2 Resuelto)
def vectorize_data(train_texts, test_texts, ngram_range, max_features, min_df):
    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features, min_df=min_df)
    X_train = vectorizer.fit_transform(train_texts)
    X_test = vectorizer.transform(test_texts)
    return X_train, X_test, vectorizer

# 3. Modelo de Regresión Softmax (TODO 3 Resuelto)
class SoftmaxRegression(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        # Una única capa lineal sin capas ocultas
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        # Devolvemos los logits crudos (CrossEntropyLoss aplica Softmax por dentro)
        return self.linear(x)

# Lógica de Entrenamiento y Carga de Cache

In [ ]:
def to_tensor_dataset(X, y):
    X_dense = torch.tensor(X.toarray(), dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.long)
    return TensorDataset(X_dense, y_tensor)


def evaluate(model, loader, criterion):
    # Pérdida y accuracy promedio de `model` sobre `loader` (sin actualizar pesos).
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            running_loss += criterion(logits, yb).item() * xb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += xb.size(0)
    return running_loss / total, correct / total


def train_softmax_model(model, train_loader, val_loader, optimizer, epochs,
                        writer=None, run_name="softmax", log_mlflow=False, verbose=True):
    # Entrena `model` y devuelve el historial por época.
    #   val_loader : DataLoader de validación, o None (entrenamiento final sobre todo el train:
    #                en ese caso no se mira ningún conjunto mientras se entrena).
    #   writer     : SummaryWriter de TensorBoard (opcional). Escribe loss/accuracy por época.
    #   log_mlflow : si es True y hay un run de MLflow activo, registra las mismas métricas
    #                con mlflow.log_metric(..., step=epoch).
    #   verbose    : imprime una línea por época.
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [],
               "epoch_time": [], "cum_time": []}
    t_start = time.perf_counter()

    for epoch in range(epochs):
        t_ep = time.perf_counter()
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += xb.size(0)
        train_loss, train_acc = running_loss / total, correct / total

        if val_loader is not None:
            val_loss, val_acc = evaluate(model, val_loader, criterion)
        else:
            val_loss, val_acc = float("nan"), float("nan")

        epoch_time = time.perf_counter() - t_ep
        history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc);   history["val_acc"].append(val_acc)
        history["epoch_time"].append(epoch_time)
        history["cum_time"].append(time.perf_counter() - t_start)

        # --- Registro de experimentos ---
        if writer is not None:
            writer.add_scalar(f"{run_name}/loss/train", train_loss, epoch + 1)
            writer.add_scalar(f"{run_name}/acc/train", train_acc, epoch + 1)
            writer.add_scalar(f"{run_name}/time/epoch_s", epoch_time, epoch + 1)
            if val_loader is not None:
                writer.add_scalar(f"{run_name}/loss/val", val_loss, epoch + 1)
                writer.add_scalar(f"{run_name}/acc/val", val_acc, epoch + 1)
        if log_mlflow and mlflow.active_run() is not None:
            mlflow.log_metric("train_loss", train_loss, step=epoch + 1)
            mlflow.log_metric("train_acc", train_acc, step=epoch + 1)
            mlflow.log_metric("epoch_time_s", epoch_time, step=epoch + 1)
            if val_loader is not None:
                mlflow.log_metric("val_loss", val_loss, step=epoch + 1)
                mlflow.log_metric("val_acc", val_acc, step=epoch + 1)
        if verbose:
            msg = f"[{run_name}] época {epoch + 1:02d}/{epochs} | train loss {train_loss:.4f} acc {train_acc:.4f}"
            if val_loader is not None:
                msg += f" | val loss {val_loss:.4f} acc {val_acc:.4f}"
            print(msg + f" | {epoch_time:.1f}s")
    return history


def prepare_loaders(p, split="val", batch_size=None):
    # Arma los DataLoaders de un conjunto de hiperparámetros `p` (dict con las claves del trial).
    #   split="val"  : mismo split train/validación 80/20 estratificado (SEED) que usa la búsqueda.
    #   split="full" : todo el train (para el entrenamiento final; el test se arma aparte).
    # Devuelve (train_loader, val_loader | None, input_dim, prep_time), donde prep_time es lo que
    # cuesta vectorizar con TF-IDF y densificar (se mide aparte del entrenamiento).
    t0 = time.perf_counter()
    ngram = (1, 1) if p["ngram_range"] == "1gram" else (1, 2)
    bs = batch_size or p["batch_size"]
    train_texts, _ = preprocessed_cache[(p["remove_stopwords"], p["normalization"])]
    if split == "val":
        tr_texts, val_texts, y_tr, y_val = train_test_split(
            train_texts, newsgroups_train.target, test_size=0.2, random_state=SEED,
            stratify=newsgroups_train.target)
        X_tr, X_val, _ = vectorize_data(tr_texts, val_texts, ngram, p["max_features"], p["min_df"])
        tr_loader = DataLoader(to_tensor_dataset(X_tr, y_tr), batch_size=bs, shuffle=True)
        val_loader = DataLoader(to_tensor_dataset(X_val, y_val), batch_size=bs, shuffle=False)
        return tr_loader, val_loader, X_tr.shape[1], time.perf_counter() - t0
    X_tr, _, _ = vectorize_data(train_texts, train_texts[:1], ngram, p["max_features"], p["min_df"])
    tr_loader = DataLoader(to_tensor_dataset(X_tr, newsgroups_train.target), batch_size=bs, shuffle=True)
    return tr_loader, None, X_tr.shape[1], time.perf_counter() - t0


print("Precalculando preprocesamiento para las 6 combinaciones (puede tardar unos minutos)...")
preprocessed_cache = {}
for remove_stopwords in [False, True]:
    for normalization in ["none", "stem", "lemmatize"]:
        key = (remove_stopwords, normalization)
        train_proc = [preprocess_text(t, remove_stopwords, normalization) for t in newsgroups_train.data]
        test_proc  = [preprocess_text(t, remove_stopwords, normalization) for t in newsgroups_test.data]
        preprocessed_cache[key] = (train_proc, test_proc)
        print("  listo:", key)

# Búsqueda de Hiperparámetros

Se usan **15 trials** (los pedidos en la consigna) de Optuna con `TPESampler(seed=42)`, de modo que la búsqueda sea reproducible. Cada trial entrena 5 épocas sobre el split 80/20 estratificado y devuelve la accuracy de validación de la última época. Cada trial queda registrado como *run anidado* de MLflow (parámetros y métricas por época) y como carpeta de TensorBoard (`runs/tp1/trial_N`). Además se miden por separado el tiempo de preparación (TF-IDF + densificado) y el de entrenamiento.

In [ ]:
N_TRIALS = 15
EPOCHS_SEARCH = 5

def objective(trial):
    remove_stopwords = trial.suggest_categorical("remove_stopwords", [False, True])
    normalization = trial.suggest_categorical("normalization", ["none", "stem", "lemmatize"])
    ngram_range = trial.suggest_categorical("ngram_range", ["1gram", "1_2gram"])
    max_features = trial.suggest_categorical("max_features", [2000, 5000, 10000])
    min_df = trial.suggest_categorical("min_df", [1, 2, 5])
    lr = trial.suggest_float("lr", 1e-3, 1e-1, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

    with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
        mlflow.log_params(trial.params)
        train_loader, val_loader, input_dim, prep_time = prepare_loaders(trial.params, split="val")

        model = SoftmaxRegression(input_dim=input_dim, num_classes=NUM_CLASSES)
        optimizer = optim.Adam(model.parameters(), lr=lr)
        writer = SummaryWriter(f"{TB_DIR}/trial_{trial.number}")

        t0 = time.perf_counter()
        history = train_softmax_model(model, train_loader, val_loader, optimizer, epochs=EPOCHS_SEARCH,
                                      writer=writer, run_name=f"trial_{trial.number}",
                                      log_mlflow=True, verbose=False)
        train_time = time.perf_counter() - t0
        writer.add_hparams({k: str(v) for k, v in trial.params.items()},
                           {"hparam/val_acc": history["val_acc"][-1]})
        writer.close()

        trial.set_user_attr("prep_time", prep_time)
        trial.set_user_attr("train_time", train_time)
        trial.set_user_attr("final_train_acc", history["train_acc"][-1])
        trial.set_user_attr("final_val_loss", history["val_loss"][-1])
        mlflow.log_metrics({"prep_time_s": prep_time, "train_time_s": train_time,
                            "final_val_acc": history["val_acc"][-1]})
    return history["val_acc"][-1]

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED),
                            study_name="tp1_softmax")
with mlflow.start_run(run_name="optuna_search"):
    study.optimize(objective, n_trials=N_TRIALS)
    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
    mlflow.log_metric("best_val_acc", study.best_value)

print("Mejores hiperparámetros:", study.best_params)
print("Mejor accuracy de validación:", study.best_value)

# Entrenamiento Final y Métricas

El test se usa **una sola vez**, al final. El orden es:

1. **Elegir las épocas con validación:** se entrena la mejor configuración hasta 30 épocas sobre el split 80/20 y se toma la época donde la *val loss* es mínima. Las curvas de esta etapa son de validación.
2. **Reentrenar sobre todo el train** ese número de épocas, sin pasar ningún `val_loader` (no se mira ningún conjunto durante el entrenamiento).
3. **Evaluar en test**: accuracy, reporte por clase y matriz de confusión.

In [ ]:
best = study.best_params
BEST_LR = best["lr"]
BEST_BATCH_SIZE = best["batch_size"]
print("Mejor configuración:", best)

# ---------- 1) Elección de épocas con la validación ----------
EPOCHS_MAX = 30
torch.manual_seed(SEED)
tr_loader, val_loader, input_dim, _ = prepare_loaders(best, split="val")
model_sel = SoftmaxRegression(input_dim, NUM_CLASSES)
opt_sel = optim.Adam(model_sel.parameters(), lr=BEST_LR)
writer_sel = SummaryWriter(f"{TB_DIR}/seleccion_epocas")
with mlflow.start_run(run_name="seleccion_epocas"):
    mlflow.log_params(best)
    hist_sel = train_softmax_model(model_sel, tr_loader, val_loader, opt_sel, epochs=EPOCHS_MAX,
                                   writer=writer_sel, run_name="seleccion", log_mlflow=True, verbose=True)
writer_sel.close()

EPOCH_BEST_LOSS = int(np.argmin(hist_sel["val_loss"])) + 1
EPOCH_BEST_ACC = int(np.argmax(hist_sel["val_acc"])) + 1
EPOCHS_FINAL = EPOCH_BEST_LOSS
print(f"\nMínimo de val loss en la época {EPOCH_BEST_LOSS} (val loss {min(hist_sel['val_loss']):.4f}, "
      f"val acc {hist_sel['val_acc'][EPOCH_BEST_LOSS - 1]:.4f})")
print(f"Máximo de val acc en la época {EPOCH_BEST_ACC} (val acc {max(hist_sel['val_acc']):.4f})")
print(f"=> EPOCHS_FINAL = {EPOCHS_FINAL} (criterio: mínimo de val loss)")

pd.DataFrame(hist_sel).assign(epoca=lambda d: d.index + 1).to_csv("results/tp1_curvas_validacion.csv", index=False)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4))
ep = range(1, EPOCHS_MAX + 1)
a1.plot(ep, hist_sel["train_loss"], label="train"); a1.plot(ep, hist_sel["val_loss"], label="validación")
a1.axvline(EPOCHS_FINAL, color="gray", ls="--", label=f"épocas elegidas ({EPOCHS_FINAL})")
a1.set_title("Loss (split train/validación)"); a1.set_xlabel("Época"); a1.legend(); a1.grid(alpha=.4)
a2.plot(ep, hist_sel["train_acc"], label="train"); a2.plot(ep, hist_sel["val_acc"], label="validación")
a2.axvline(EPOCHS_FINAL, color="gray", ls="--")
a2.set_title("Accuracy (split train/validación)"); a2.set_xlabel("Época"); a2.legend(); a2.grid(alpha=.4)
plt.tight_layout(); plt.savefig("figures/tp1_curvas_validacion.png", dpi=150); plt.show()

# ---------- 2) Reentrenar sobre TODO el train, sin val_loader ----------
torch.manual_seed(SEED)
full_loader, _, input_dim_full, _ = prepare_loaders(best, split="full")
ngram_best = (1, 1) if best["ngram_range"] == "1gram" else (1, 2)
train_texts_final, test_texts_final = preprocessed_cache[(best["remove_stopwords"], best["normalization"])]
X_train_final, X_test_final, vectorizer_final = vectorize_data(
    train_texts_final, test_texts_final, ngram_best, best["max_features"], best["min_df"])
train_loader_final = DataLoader(to_tensor_dataset(X_train_final, newsgroups_train.target),
                                batch_size=BEST_BATCH_SIZE, shuffle=True)
test_loader_final = DataLoader(to_tensor_dataset(X_test_final, newsgroups_test.target),
                               batch_size=BEST_BATCH_SIZE, shuffle=False)

model = SoftmaxRegression(input_dim=X_train_final.shape[1], num_classes=NUM_CLASSES)
optimizer = optim.Adam(model.parameters(), lr=BEST_LR)
writer_final = SummaryWriter(f"{TB_DIR}/final")

print("\n--- Entrenando modelo final sobre todo el train (sin mirar el test) ---")
with mlflow.start_run(run_name="entrenamiento_final"):
    mlflow.log_params({**best, "epochs_final": EPOCHS_FINAL})
    hist_final = train_softmax_model(model, train_loader_final, None, optimizer, epochs=EPOCHS_FINAL,
                                     writer=writer_final, run_name="final", log_mlflow=True, verbose=True)

    # ---------- 3) Evaluación en test (primer y único uso del test) ----------
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in test_loader_final:
            all_preds.extend(model(xb.to(device)).argmax(dim=1).cpu().numpy())
            all_labels.extend(yb.numpy())
    TEST_ACC = accuracy_score(all_labels, all_preds)
    mlflow.log_metric("test_acc", TEST_ACC)
writer_final.add_scalar("final/test_acc", TEST_ACC, EPOCHS_FINAL)
writer_final.close()
print(f"\nAccuracy en test final: {TEST_ACC:.4f}\n")

pd.DataFrame(hist_final).assign(epoca=lambda d: d.index + 1).to_csv("results/tp1_curvas_entrenamiento_final.csv", index=False)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4))
a1.plot(range(1, EPOCHS_FINAL + 1), hist_final["train_loss"], marker="o")
a1.set_title("Entrenamiento final: loss de train"); a1.set_xlabel("Época"); a1.grid(alpha=.4)
a2.plot(range(1, EPOCHS_FINAL + 1), hist_final["train_acc"], marker="o")
a2.set_title("Entrenamiento final: accuracy de train"); a2.set_xlabel("Época"); a2.grid(alpha=.4)
plt.tight_layout(); plt.savefig("figures/tp1_curvas_entrenamiento_final.png", dpi=150); plt.show()

# Reporte por clase
print(classification_report(all_labels, all_preds, target_names=class_names, digits=3))
report_df = pd.DataFrame(classification_report(all_labels, all_preds, target_names=class_names,
                                               output_dict=True)).T
report_df.to_csv("results/tp1_reporte_por_clase.csv")

# Matriz de confusión
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(cm_norm, cmap="Blues")
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(class_names, rotation=90, fontsize=8)
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(class_names, fontsize=8)
ax.set_xlabel("Predicción"); ax.set_ylabel("Clase real"); ax.set_title("Matriz de confusión (normalizada por fila) — test")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.savefig("figures/tp1_matriz_confusion.png", dpi=150); plt.show()

# Pares de clases más confundidos (fuera de la diagonal)
off = cm.copy(); np.fill_diagonal(off, 0)
top = np.dstack(np.unravel_index(np.argsort(-off.ravel())[:6], off.shape))[0]
print("Confusiones más frecuentes (clase real -> predicha):")
for i, j in top:
    print(f"  {class_names[i]:28s} -> {class_names[j]:28s} {off[i, j]:4d} docs ({cm_norm[i, j]:.1%} de la clase)")
f1 = report_df.loc[class_names, "f1-score"].sort_values()
print(f"\nClases con menor F1: " + ", ".join(f"{c} ({v:.2f})" for c, v in f1.head(3).items()))
print(f"Clases con mayor F1: " + ", ".join(f"{c} ({v:.2f})" for c, v in f1.tail(3).items()))

**Nota sobre las curvas.** Las curvas con validación (arriba) salen del split 80/20 y son las que se usan para decidir las épocas. Las curvas del entrenamiento final solo tienen *train*, porque el modelo final se entrena sobre todo el train y el test no se mira hasta la evaluación. Todas se guardan en `figures/` y `results/`.

# Bias y varianza

Para diagnosticar si el modelo está **subajustado** (sesgo alto) o **sobreajustado** (varianza alta) se compara el error en train con el de validación, en la época elegida y al final del entrenamiento, y con el error en test. Los umbrales usados para el diagnóstico automático son orientativos (error de train > 15 % → sesgo alto; brecha train–validación > 10 puntos → varianza alta).

In [ ]:
k = EPOCHS_FINAL - 1
rows = []
for etiqueta, idx in [(f"Época elegida ({EPOCHS_FINAL})", k), (f"Época {EPOCHS_MAX} (sin parar)", EPOCHS_MAX - 1)]:
    rows.append(dict(punto=etiqueta,
                     train_acc=hist_sel["train_acc"][idx], val_acc=hist_sel["val_acc"][idx],
                     train_loss=hist_sel["train_loss"][idx], val_loss=hist_sel["val_loss"][idx]))
bv = pd.DataFrame(rows)
bv["err_train"] = 1 - bv.train_acc
bv["err_val"] = 1 - bv.val_acc
bv["brecha_acc"] = bv.train_acc - bv.val_acc
display(bv.round(4))
print(f"Error en test del modelo final: {1 - TEST_ACC:.4f}")

r = bv.iloc[0]
sesgo = "ALTO" if r.err_train > 0.15 else "bajo"
varianza = "ALTA" if r.brecha_acc > 0.10 else "baja"
diag = []
if r.err_train > 0.15: diag.append("subajuste (el modelo ni siquiera ajusta bien el train)")
if r.brecha_acc > 0.10: diag.append("sobreajuste (se aprende bien el train pero no generaliza igual)")
display(Markdown(
    f"**Diagnóstico en la época elegida ({EPOCHS_FINAL}):** error de train {r.err_train:.3f}, error de validación {r.err_val:.3f} "
    f"(brecha de {r.brecha_acc * 100:.1f} puntos de accuracy) y error de test {1 - TEST_ACC:.3f}. "
    f"Sesgo **{sesgo}**, varianza **{varianza}**" + (f" → {' y '.join(diag)}." if diag else ".") + "  \n"
    f"Si se sigue entrenando hasta la época {EPOCHS_MAX}, la brecha pasa a {bv.iloc[1].brecha_acc * 100:.1f} puntos y la val loss "
    f"pasa de {r.val_loss:.3f} a {bv.iloc[1].val_loss:.3f}"
    + (": la loss de validación sube mientras la de train sigue bajando, es decir, más entrenamiento aumenta la varianza."
       if bv.iloc[1].val_loss > r.val_loss else ": todavía no hay sobreajuste visible en la loss de validación.")))

**Cómo leer este diagnóstico.** Un modelo lineal sobre TF-IDF con miles de features tiene mucha capacidad *relativa* al tamaño del dataset (≈11 mil documentos), por eso suele alcanzar un error de train bajo (sesgo bajo) y mostrar una brecha grande con validación (varianza alta). Las palancas para reducir la varianza son más datos, menos features (`max_features`, `min_df`), regularización (weight decay / L2) y parar antes (early stopping, que es lo que hace la elección de épocas de arriba). Reducir el sesgo, en cambio, requeriría más capacidad (un MLP, que se estudia en el TP2).

# Explicación del Código (Consigna 1)

El pipeline completo está organizado en bloques; cada uno corresponde a una celda del notebook.

**1. Carga de datos y configuración (celda de imports).** Se carga 20 Newsgroups (≈11.3 mil documentos de train y 7.5 mil de test, 20 clases) con `fetch_20newsgroups`, quitando encabezados, firmas y citas (`remove=("headers","footers","quotes")`) para que el modelo no se apoye en metadatos que delatan el grupo. Se fijan las semillas (`random`, `numpy`, `torch`) y el dispositivo (CPU/GPU). Se inicializan las carpetas de salida y el tracking de MLflow.

**2. Preprocesamiento (`preprocess_text`, NLTK).** Pasa a minúsculas, tokeniza con `word_tokenize`, se queda solo con tokens alfabéticos y, según los hiperparámetros, elimina *stopwords* y normaliza con *stemming* (Porter: recorta sufijos, "running" → "run") o *lematización* (WordNet: devuelve la forma de diccionario). Como el preprocesamiento es lento, se **precalcula una vez por cada una de las 6 combinaciones** (stopwords × normalización) y se guarda en `preprocessed_cache`; así Optuna no repite ese trabajo en cada trial.

**3. Vectorización (`vectorize_data`, scikit-learn).** `TfidfVectorizer` convierte cada documento en un vector donde cada coordenada pondera una palabra (o bigrama) por su frecuencia en el documento y su rareza en el corpus (TF-IDF). Se ajusta (`fit`) **solo con el train** y se aplica (`transform`) a validación o test, para no filtrar información. Hiperparámetros: `ngram_range` (unigramas o uni+bigramas), `max_features` (tamaño del vocabulario) y `min_df` (frecuencia mínima de documentos). `to_tensor_dataset` pasa la matriz esparsa a tensores densos de PyTorch.

**4. Modelo (`SoftmaxRegression`, PyTorch).** Una única capa lineal `nn.Linear(input_dim, 20)`: calcula un puntaje (logit) por clase. No aplica softmax en `forward` porque `nn.CrossEntropyLoss` ya combina `log_softmax` + log-verosimilitud negativa, que es numéricamente más estable. La predicción es el `argmax` de los logits. El optimizador es Adam.

**5. Entrenamiento (`train_softmax_model`).** Por cada época recorre el train en mini-batches (forward → loss → `backward` → `optimizer.step`), acumula loss y accuracy, y luego evalúa en validación sin gradientes (`model.eval()` + `torch.no_grad()`). Devuelve el historial por época (loss, accuracy y tiempo). Si se le pasa `val_loader=None` solo entrena. En cada época además:
- imprime una línea de progreso (`verbose`),
- escribe las métricas en **TensorBoard** (`SummaryWriter.add_scalar`), y
- las registra en **MLflow** (`mlflow.log_metric(..., step=epoch)`).

**6. Búsqueda de hiperparámetros (`objective` + Optuna).** Optuna es un framework de optimización de hiperparámetros. Cada *trial* muestrea una configuración (`suggest_*`), entrena y devuelve la accuracy de validación; Optuna usa el sampler **TPE** (un modelo bayesiano que usa los trials anteriores para proponer configuraciones prometedoras) en lugar de una búsqueda exhaustiva en grilla, que con 6 × 2 × 3 × 3 × 3 × 3 combinaciones discretas más un lr continuo sería inviable. La búsqueda combina hiperparámetros de NLP (stopwords, normalización, n-gramas, vocabulario, `min_df`) con los de la red (lr, batch size). Se usan 15 trials y semilla fija.

**7. Registro de experimentos: rol de TensorBoard y MLflow.**
- **TensorBoard** es un visor de curvas en vivo: cada trial escribe su loss/accuracy por época en `runs/tp1/trial_N`; se abre con `tensorboard --logdir runs` y permite comparar trials superponiendo curvas y ver la tabla de hiperparámetros (`add_hparams`).
- **MLflow** es un registro de experimentos: guarda, para cada ejecución, los hiperparámetros, las métricas y los tiempos en una base SQLite local (`mlflow.db`); se explora con `mlflow ui`. Acá hay un run padre (`optuna_search`) con un run anidado por trial, más los runs de selección de épocas y de entrenamiento final. Sirve para reproducir y comparar corridas sin depender de lo impreso en el notebook.

**8. Entrenamiento final y evaluación.** Se elige el número de épocas con la validación, se reentrena con todo el train y se evalúa **una vez** en test con accuracy, reporte por clase (precision/recall/F1) y matriz de confusión.

# Análisis de Hiperparámetros (Consigna 2)

In [ ]:
display(vis.plot_param_importances(study))
display(vis.plot_optimization_history(study))
plt.savefig("figures/tp1_historia_optimizacion.png", dpi=150)

def trials_df(st):
    rows = []
    for t in st.trials:
        if t.value is None:
            continue
        rows.append(dict(trial=t.number, val_acc=t.value, **t.params,
                         prep_time=t.user_attrs.get("prep_time"), train_time=t.user_attrs.get("train_time"),
                         train_acc=t.user_attrs.get("final_train_acc")))
    return pd.DataFrame(rows)

df = trials_df(study)
df["log10_lr"] = np.log10(df.lr)
df.to_csv("results/tp1_trials.csv", index=False)
print(f"Trials completados: {len(df)}")
display(df.sort_values("val_acc", ascending=False).round(4).reset_index(drop=True))

for col in ["remove_stopwords", "normalization", "ngram_range", "max_features", "min_df", "batch_size"]:
    print(f"\n=== val_acc por {col} ===")
    print(df.groupby(col).val_acc.agg(n="count", media="mean", max="max").round(4))

In [ ]:
N_VAL = int(0.2 * len(newsgroups_train.data))        # tamaño del conjunto de validación
SE = (0.7 * 0.3 / N_VAL) ** 0.5                         # error estándar aproximado de una accuracy ~0.7

def stat(mask):
    m = df[mask]
    return f"n={len(m)}, media {m.val_acc.mean():.3f}" if len(m) else "sin trials en ese rango"

b = df.loc[df.val_acc.idxmax()]
rho = df[["log10_lr", "val_acc"]].corr(method="spearman").iloc[0, 1]
sw = df.groupby("remove_stopwords").val_acc.mean()
vocab = df.groupby("max_features").val_acc.mean()
ng = df.groupby("ngram_range").val_acc.mean()
norm_ = df.groupby("normalization").val_acc.mean()
bs = df.groupby("batch_size").val_acc.mean()

lineas = [
    f"**Learning rate.** El mejor trial (#{int(b.trial)}) usó lr = {b.lr:.4f} y llegó a {b.val_acc:.4f}. "
    f"Por rango: lr < 0.005 → {stat(df.lr < 0.005)}; 0.005 ≤ lr < 0.03 → {stat((df.lr >= 0.005) & (df.lr < 0.03))}; "
    f"lr ≥ 0.03 → {stat(df.lr >= 0.03)}. Correlación de Spearman entre log(lr) y val_acc: {rho:+.2f}.",
    f"**Stopwords.** Con stopwords removidas la accuracy media fue " +
    (f"{sw.get(True, float('nan')):.3f} contra {sw.get(False, float('nan')):.3f} sin removerlas "
     f"(diferencia de {(sw.get(True, np.nan) - sw.get(False, np.nan)) * 100:+.1f} puntos; el error estándar de la validación es ≈ ±{SE * 100:.1f} puntos)."),
    f"**Vocabulario (`max_features`).** Accuracy media por tamaño: " + ", ".join(f"{int(k)} → {v:.3f}" for k, v in vocab.items()) + ".",
    f"**N-gramas.** " + ", ".join(f"{k} → {v:.3f}" for k, v in ng.items()) + ".",
    f"**Normalización.** " + ", ".join(f"{k} → {v:.3f}" for k, v in norm_.items()) + ".",
    f"**Batch size.** " + ", ".join(f"{int(k)} → {v:.3f}" for k, v in bs.items()) + ".",
]
display(Markdown("### Resumen numérico de la búsqueda\n\n" + "\n\n".join("- " + l for l in lineas)))

**Cómo interpretar estos números.**

- Con solo 15 trials, 7 hiperparámetros y un error estándar de la validación de ≈ ±1 punto, **las diferencias menores a ~1 punto no son concluyentes**; los promedios por valor mezclan el efecto de los demás hiperparámetros, porque TPE no muestrea de forma balanceada.
- La **importancia de hiperparámetros** del gráfico (fANOVA/PED-ANOVA) se estima con muy pocos puntos, por lo que sirve como orientación, no como medición.
- El learning rate es el único hiperparámetro continuo y es el que más condiciona si el modelo llega a converger en las 5 épocas de cada trial: con un lr muy bajo el modelo no alcanza a aprender y con uno muy alto la optimización se vuelve ruidosa. Los rangos y valores concretos están en la tabla de arriba.
- Las decisiones de preprocesamiento y vocabulario (stopwords, n-gramas, `max_features`) actúan sobre la **calidad de las features**: más vocabulario aporta palabras discriminativas, pero con datos limitados también aumenta la varianza (esto se ve en la sección de dinámica).

# Dinámica de los Entrenamientos (Consigna 3)

Para que el texto describa solo lo que se ve, se entrena (15 épocas, **mismo split train/validación que la búsqueda**) un trial por cada comportamiento que se quiere analizar:

- **Mejor** y **peor** trial de la búsqueda.
- El trial de **lr más bajo** (aprendizaje lento).
- El trial de **lr más alto** (posible inestabilidad).
- Un trial con **bigramas y el vocabulario más grande** disponible (posible sobreajuste por vocabulario).

Las curvas de *validación* son las del split de validación (no del test). Si un mismo trial cumple varios roles, se grafica una sola vez.

In [ ]:
EPOCHS_DYN = 15
ts = [t for t in study.trials if t.value is not None]
roles = {}
def add_role(label, t): roles.setdefault(t.number, []).append(label)

add_role("Mejor trial", study.best_trial)
add_role("Peor trial", min(ts, key=lambda t: t.value))
add_role("lr más bajo", min(ts, key=lambda t: t.params["lr"]))
add_role("lr más alto", max(ts, key=lambda t: t.params["lr"]))
bi = [t for t in ts if t.params["ngram_range"] == "1_2gram"]
if bi:
    add_role("Bigramas + vocabulario grande", max(bi, key=lambda t: (t.params["max_features"], t.value)))
else:
    print("Ningún trial usó bigramas: el comportamiento 'overfitting por bigramas' no se puede mostrar con esta búsqueda.")

dyn = {}
for num, labels in roles.items():
    t = study.trials[num]
    torch.manual_seed(SEED)
    trl, vl, d, prep = prepare_loaders(t.params, split="val")
    m = SoftmaxRegression(d, NUM_CLASSES)
    o = optim.Adam(m.parameters(), lr=t.params["lr"])
    h = train_softmax_model(m, trl, vl, o, epochs=EPOCHS_DYN, run_name=f"dyn_{num}", verbose=False)
    dyn[num] = dict(labels=labels, trial=t, hist=h, prep=prep)
    print(f"trial {num} ({' / '.join(labels)}): listo")

fig, axes = plt.subplots(len(dyn), 2, figsize=(14, 3.6 * len(dyn)), squeeze=False)
for row, (num, r) in zip(axes, dyn.items()):
    h, p = r["hist"], r["trial"].params
    ep = range(1, EPOCHS_DYN + 1)
    ttl = f"Trial {num} — {' / '.join(r['labels'])}\nlr={p['lr']:.4f}, {p['ngram_range']}, {p['max_features']} feat., batch {p['batch_size']}"
    row[0].plot(ep, h["train_loss"], marker="o", ms=3, label="Train"); row[0].plot(ep, h["val_loss"], marker="o", ms=3, label="Validación")
    row[0].set_title(ttl + " — Loss", fontsize=9); row[0].set_xlabel("Época"); row[0].legend(); row[0].grid(alpha=.4)
    row[1].plot(ep, h["train_acc"], marker="o", ms=3, label="Train"); row[1].plot(ep, h["val_acc"], marker="o", ms=3, label="Validación")
    row[1].axvline(EPOCHS_SEARCH, color="gray", ls=":", label=f"fin de la búsqueda ({EPOCHS_SEARCH} ép.)")
    row[1].set_title("Accuracy", fontsize=9); row[1].set_xlabel("Época"); row[1].legend(); row[1].grid(alpha=.4)
plt.tight_layout(); plt.savefig("figures/tp1_dinamica_trials.png", dpi=150); plt.show()

In [ ]:
def metricas_dinamica(h):
    vl = np.array(h["val_loss"]); va = np.array(h["val_acc"]); ta = np.array(h["train_acc"])
    dl = np.diff(vl)
    return dict(
        train_acc=ta[-1], val_acc=va[-1], brecha=ta[-1] - va[-1],
        val_acc_ep5=va[EPOCHS_SEARCH - 1],
        val_loss_min=vl.min(), ep_min_val_loss=int(vl.argmin()) + 1, val_loss_final=vl[-1],
        subida_val_loss=(vl[-1] / vl.min() - 1),
        oscilacion=float(np.std(dl[1:])) if len(dl) > 2 else float("nan"),
        mejora_ult5=va[-1] - va[-6],
    )

filas = []
for num, r in dyn.items():
    p = r["trial"].params
    filas.append(dict(trial=num, rol=" / ".join(r["labels"]), lr=p["lr"], ngram=p["ngram_range"],
                      max_features=p["max_features"], batch=p["batch_size"],
                      val_optuna=r["trial"].value, **metricas_dinamica(r["hist"])))
dyn_df = pd.DataFrame(filas)
dyn_df.to_csv("results/tp1_dinamica_metricas.csv", index=False)
display(dyn_df.round(4).set_index("trial"))

lineas = []
for _, r in dyn_df.iterrows():
    obs = []
    obs.append(f"train acc {r.train_acc:.3f} vs val acc {r.val_acc:.3f} (brecha {r.brecha * 100:.1f} pts) a las {EPOCHS_DYN} épocas")
    obs.append(f"val loss mínima {r.val_loss_min:.3f} en la época {int(r.ep_min_val_loss)}, final {r.val_loss_final:.3f}")
    if r.subida_val_loss > 0.03:
        obs.append(f"la val loss **vuelve a subir {r.subida_val_loss:.0%}** respecto de su mínimo mientras la de train baja → sobreajuste")
    if r.mejora_ult5 > 0.03:
        obs.append(f"la val acc todavía mejoró {r.mejora_ult5 * 100:.1f} pts en las últimas 5 épocas → **no había convergido**")
    if r.oscilacion > 0.05:
        obs.append(f"la val loss es **inestable** (desvío de sus cambios entre épocas = {r.oscilacion:.3f})")
    obs.append(f"val acc en la época {EPOCHS_SEARCH}: {r.val_acc_ep5:.3f} (Optuna reportó {r.val_optuna:.3f} para este trial)")
    lineas.append(f"- **Trial {int(r.trial)} — {r.rol}** (lr={r.lr:.4f}, {r.ngram}, {int(r.max_features)} features, batch {int(r.batch)}): " + "; ".join(obs) + ".")
display(Markdown("### Lectura de las curvas (generada con las métricas de arriba)\n\n" + "\n".join(lineas)))

**Cómo leer las curvas.** Con regresión softmax sobre TF-IDF la *train loss* baja casi monótonamente; lo que distingue los casos es la forma de la *val loss* y la brecha de accuracy:

- **lr bajo:** los pasos son chicos, así que train y validación bajan lentamente y casi pegadas; si la accuracy todavía crece en las últimas épocas, el modelo no convergió dentro del presupuesto de épocas (la tabla marca "no había convergido" cuando la mejora en las últimas 5 épocas supera 3 puntos).
- **lr alto:** los pasos grandes aceleran la bajada inicial pero la loss de validación tiende a ser más ruidosa; en un modelo lineal convexo con Adam el efecto suele ser moderado (la optimización no *diverge* del todo), por eso el texto solo habla de inestabilidad si la métrica de oscilación la detecta.
- **Bigramas + vocabulario grande:** aumentan fuertemente el número de parámetros (20 × `max_features`) respecto de los datos disponibles; se espera train acc más alta y una brecha mayor con validación. La tabla permite verificarlo contra los otros casos.

Las frases automáticas de arriba solo afirman un comportamiento cuando la métrica lo muestra; si alguno de los tres no aparece con esta búsqueda, no se afirma.

# Tiempos de Convergencia (Consigna 4)

Se mide de tres maneras:

1. **Duración por trial de Optuna**, separando *preparación* (TF-IDF + densificado) de *entrenamiento*, para que la vectorización no se confunda con el costo del modelo.
2. **Tiempo por época** y **tiempo hasta converger** en un experimento controlado: partiendo de la mejor configuración se cambia **un** hiperparámetro por vez (batch size, n-gramas, tamaño de vocabulario, lr) y se entrena 15 épocas sobre el mismo split.
3. **Criterio de convergencia:** tiempo (y época) en que la val accuracy alcanza por primera vez el **95 % de la mejor val accuracy de la configuración base** (objetivo común para comparar todas las configuraciones). Si nunca lo alcanza se marca como "no alcanza".

In [ ]:
df_t = df.copy()
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(df_t.trial, df_t.prep_time, label="Preparación (TF-IDF + densificado)", color="orange", alpha=.8)
ax.bar(df_t.trial, df_t.train_time, bottom=df_t.prep_time, label=f"Entrenamiento ({EPOCHS_SEARCH} épocas)", color="teal", alpha=.8)
ax.set_xlabel("Trial"); ax.set_ylabel("Segundos"); ax.set_xticks(df_t.trial)
ax.set_title("Duración de cada trial de Optuna, separada en preparación y entrenamiento")
ax.legend(); ax.grid(axis="y", alpha=.4)
plt.tight_layout(); plt.savefig("figures/tp1_tiempos_trials.png", dpi=150); plt.show()

df_t["epoca_s"] = df_t.train_time / EPOCHS_SEARCH
print("Tiempo por época de entrenamiento (s) y preparación (s) según el hiperparámetro:")
for col in ["batch_size", "ngram_range", "max_features"]:
    print(f"\n--- por {col} ---")
    print(df_t.groupby(col).agg(n=("trial", "count"), epoca_s=("epoca_s", "median"), prep_s=("prep_time", "median")).round(2))

In [ ]:
# ---- Experimento controlado: un cambio por vez respecto de la mejor configuración ----
base = dict(best)
variantes = [("base (mejor trial)", {})]
for bsz in [32, 64, 128]:
    if bsz != base["batch_size"]:
        variantes.append((f"batch_size={bsz}", {"batch_size": bsz}))
variantes.append(("1_2gram (bigramas)" if base["ngram_range"] == "1gram" else "1gram (unigramas)",
                  {"ngram_range": "1_2gram" if base["ngram_range"] == "1gram" else "1gram"}))
for mf in [2000, 5000, 10000]:
    if mf != base["max_features"]:
        variantes.append((f"max_features={mf}", {"max_features": mf}))
for fac in [0.1, 10]:
    variantes.append((f"lr x{fac:g} ({base['lr'] * fac:.4f})", {"lr": base["lr"] * fac}))

EPOCHS_CONV = 15
resultados = []
for nombre, cambio in variantes:
    p = {**base, **cambio}
    torch.manual_seed(SEED)
    trl, vl, d, prep = prepare_loaders(p, split="val")
    m = SoftmaxRegression(d, NUM_CLASSES)
    o = optim.Adam(m.parameters(), lr=p["lr"])
    h = train_softmax_model(m, trl, vl, o, epochs=EPOCHS_CONV, run_name="conv", verbose=False)
    resultados.append(dict(config=nombre, prep_s=prep, epoca_s=float(np.mean(h["epoch_time"])),
                           val_acc_max=max(h["val_acc"]), hist=h))
    print(f"{nombre:28s} prep {prep:5.1f}s | {np.mean(h['epoch_time']):5.2f} s/época | val acc máx {max(h['val_acc']):.4f}")

OBJETIVO = 0.95 * resultados[0]["val_acc_max"]
for r in resultados:
    h = r["hist"]
    idx = next((i for i, a in enumerate(h["val_acc"]) if a >= OBJETIVO), None)
    r["epocas_a_objetivo"] = idx + 1 if idx is not None else np.nan
    r["t_a_objetivo_s"] = h["cum_time"][idx] if idx is not None else np.nan
conv = pd.DataFrame([{k: v for k, v in r.items() if k != "hist"} for r in resultados])
conv["tiempo_total_s"] = conv.prep_s + conv.t_a_objetivo_s          # incluye la vectorización
conv.to_csv("results/tp1_tiempos_convergencia.csv", index=False)
print(f"\nObjetivo común: val acc ≥ {OBJETIVO:.4f} (95 % de la mejor val acc de la base, {resultados[0]['val_acc_max']:.4f})")
display(conv.round(3).set_index("config"))

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
x = np.arange(len(conv))
ax[0].bar(x, conv.epoca_s, color="teal"); ax[0].set_title("Tiempo medio por época (s)")
ax[1].bar(x, conv.t_a_objetivo_s.fillna(0), color="slateblue"); ax[1].set_title(f"Tiempo de entrenamiento hasta val acc ≥ {OBJETIVO:.3f} (s)")
ax[2].bar(x, conv.prep_s, color="orange"); ax[2].set_title("Preparación: TF-IDF + densificado (s)")
for a in ax:
    a.set_xticks(x); a.set_xticklabels(conv.config, rotation=60, ha="right", fontsize=8); a.grid(axis="y", alpha=.4)
plt.tight_layout(); plt.savefig("figures/tp1_tiempos_convergencia.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(9, 4.5))
for r in resultados:
    ax.plot(r["hist"]["cum_time"], r["hist"]["val_acc"], marker=".", label=r["config"])
ax.axhline(OBJETIVO, color="k", ls=":", label="objetivo")
ax.set_xlabel("Tiempo de entrenamiento acumulado (s)"); ax.set_ylabel("Val accuracy")
ax.set_title("Val accuracy vs tiempo"); ax.legend(fontsize=7, ncol=2); ax.grid(alpha=.4)
plt.tight_layout(); plt.savefig("figures/tp1_val_vs_tiempo.png", dpi=150); plt.show()

# ---- Texto con cifras ----
g = conv.set_index("config")
b0 = g.iloc[0]
partes = []
bs_rows = [c for c in g.index if c.startswith("batch_size=")]
if bs_rows:
    partes.append("**Batch size.** Tiempo por época: " + ", ".join(
        f"{c} → {g.loc[c, 'epoca_s']:.2f} s" for c in bs_rows) + f" (base, batch {base['batch_size']}: {b0.epoca_s:.2f} s). "
        + "Tiempo hasta el objetivo: " + ", ".join(
            f"{c} → " + (f"{g.loc[c, 't_a_objetivo_s']:.1f} s" if not np.isnan(g.loc[c, 't_a_objetivo_s']) else "no alcanza") for c in bs_rows)
        + f" (base: {b0.t_a_objetivo_s:.1f} s).")
ng_row = [c for c in g.index if "gram" in c and c != g.index[0]]
if ng_row:
    c = ng_row[0]
    partes.append(f"**N-gramas.** `{c}`: preparación {g.loc[c, 'prep_s']:.1f} s contra {b0.prep_s:.1f} s de la base ({g.loc[c, 'prep_s'] / b0.prep_s:.1f}×) "
                  f"y {g.loc[c, 'epoca_s']:.2f} s por época contra {b0.epoca_s:.2f} s.")
mf_rows = [c for c in g.index if c.startswith("max_features=")]
if mf_rows:
    partes.append("**Vocabulario.** Tiempo por época: " + ", ".join(f"{c} → {g.loc[c, 'epoca_s']:.2f} s" for c in mf_rows)
                  + f" (base, {base['max_features']} features: {b0.epoca_s:.2f} s).")
lr_rows = [c for c in g.index if c.startswith("lr x")]
partes.append("**Learning rate.** Tiempo hasta el objetivo: " + ", ".join(
    f"{c} → " + (f"{g.loc[c, 't_a_objetivo_s']:.1f} s ({int(g.loc[c, 'epocas_a_objetivo'])} épocas)" if not np.isnan(g.loc[c, 't_a_objetivo_s']) else f"no alcanza en {EPOCHS_CONV} épocas")
    for c in lr_rows) + f" (base: {b0.t_a_objetivo_s:.1f} s, {int(b0.epocas_a_objetivo)} épocas).")
display(Markdown("### Análisis de tiempos (con cifras de esta corrida)\n\n" + "\n\n".join("- " + p for p in partes)))

**Qué explica los tiempos.** El costo de una época es (nº de mini-batches) × (costo de forward + backward + actualización). Un batch más chico implica más actualizaciones por época y menos paralelismo en cada una; un vocabulario más grande o los bigramas agrandan la entrada de la capa lineal (`input_dim × 20` parámetros) y por lo tanto cada multiplicación matricial. La **preparación** (TF-IDF + densificado) crece con los bigramas porque hay que contar y ordenar muchos más n-gramas. El tiempo hasta converger combina ambos efectos: depende del costo por época y de cuántas épocas hacen falta, que a su vez dependen del lr. Los tiempos son de CPU y de una sola corrida por configuración, así que diferencias de pocos puntos porcentuales pueden ser ruido de la máquina.

# Sección 9 — Preguntas de reflexión

> **Aviso para la corrección:** estas son las 10 preguntas de reflexión de la sección 9. Si el enunciado exacto del notebook base difiere en la redacción, las respuestas cubren los mismos temas del TP (regresión softmax, TF-IDF, regularización, hiperparámetros, métricas, sesgo/varianza y tiempos).

**1. ¿Por qué se usa softmax y no una sigmoide por clase o una regresión lineal para este problema?**
El problema es de clasificación multiclase con clases mutuamente excluyentes (cada documento pertenece a un único grupo). Softmax convierte los 20 puntajes en una distribución de probabilidad que suma 1, lo que modela justamente esa exclusión; con sigmoides independientes las probabilidades no suman 1 y las clases competirían sin coordinación, y una regresión lineal sobre etiquetas numéricas impondría un orden y distancias entre clases que no existen. Además, softmax + entropía cruzada tiene un gradiente simple y bien comportado (`p − y`).

**2. ¿Por qué `nn.CrossEntropyLoss` recibe logits y el modelo no aplica softmax en `forward`?**
`CrossEntropyLoss` combina `log_softmax` y la log-verosimilitud negativa en una sola operación estable (usa el truco log-sum-exp), evitando desbordes numéricos de `exp` y logaritmos de probabilidades casi cero. Aplicar softmax antes y luego la loss duplicaría la operación y sería menos estable. Para predecir basta el `argmax` de los logits, que no cambia con softmax.

**3. ¿Qué aporta TF-IDF respecto de un simple conteo de palabras?**
El conteo (bolsa de palabras) da el mismo peso a palabras muy comunes y a las informativas. TF-IDF multiplica la frecuencia en el documento por un factor inverso a en cuántos documentos aparece la palabra, de modo que términos frecuentes en todo el corpus ("the") pesan poco y términos característicos de un tema ("hockey", "god") pesan más. Además normaliza la longitud de los documentos, lo que ayuda a un modelo lineal.

**4. ¿Qué se pierde al usar bolsa de palabras / TF-IDF?**
Se pierde el orden y el contexto: "no es bueno" y "es bueno" quedan casi iguales, y no se capturan sinónimos ni polisemia. Los bigramas recuperan algo de contexto local, pero a costa de una entrada mucho más grande y esparsa (más varianza con pocos datos).

**5. ¿Por qué el vectorizador se ajusta solo con el train y no con todo el conjunto?**
Ajustar el vocabulario y los pesos IDF con validación o test filtraría información del conjunto de evaluación hacia el entrenamiento (*data leakage*), y la estimación de rendimiento sería optimista. En el notebook `fit_transform` se aplica al train y `transform` a validación/test.

**6. ¿Qué diferencia hay entre stemming y lematización, y cuándo conviene cada uno?**
El *stemming* (Porter) recorta sufijos con reglas y es rápido pero puede producir raíces que no son palabras ("studi"); la *lematización* usa un diccionario (WordNet) y devuelve la forma canónica ("studies" → "study"), es más precisa pero más lenta. Ambos reducen el tamaño del vocabulario y agrupan variantes; el efecto sobre la accuracy hay que medirlo (ver la tabla por `normalization` de la búsqueda).

**7. ¿Por qué se separa un conjunto de validación y por qué el test se usa una sola vez?**
La validación sirve para elegir hiperparámetros y épocas; si esas decisiones se tomaran mirando el test, el test dejaría de ser una estimación imparcial del rendimiento en datos nuevos (se "sobreajusta la selección"). Por eso las curvas de dinámica son de validación y el test se evalúa una única vez con el modelo final.

**8. ¿Qué papel cumple el learning rate y qué pasa si es muy chico o muy grande?**
Controla el tamaño de cada paso del optimizador. Muy chico → la loss baja lentamente y el modelo puede no converger en el presupuesto de épocas; muy grande → los pasos se pasan del mínimo y la loss oscila o diverge. Por eso es el hiperparámetro continuo más importante y se busca en escala logarítmica.

**9. ¿Cómo se diagnostica sesgo alto o varianza alta, y qué se puede hacer en cada caso?**
Comparando el error de train con el de validación: error de train alto → sesgo alto (subajuste) → más capacidad o más features; error de train bajo pero brecha grande con validación → varianza alta (sobreajuste) → más datos, regularización, menos features o parar antes. En este modelo se mide en la sección *Bias y varianza*.

**10. ¿Qué limitaciones tiene la comparación entre configuraciones en este TP?**
(a) La validación tiene unos 2263 documentos, por lo que la accuracy tiene un error estándar de ≈ ±1 punto y diferencias menores no son concluyentes; (b) con 15 trials y TPE los grupos de hiperparámetros quedan desbalanceados, y los promedios por valor están confundidos entre sí; (c) cada configuración se entrenó con una sola semilla; (d) los tiempos son de CPU y de una corrida; (e) la accuracy final en test suele ser menor que la de validación, lo que sugiere que el split de test de 20 Newsgroups tiene una distribución algo distinta.